### Imports + setup

In [ ]:
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

os.chdir("..")
from src.engine.profit import (
    expected_profit_per_loan,
    portfolio_profit_curve,
    profit_params,
    realized_profit_per_loan,
)
with open("config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

INTEREST_REVENUE, LOSS_AMOUNT, SERVICE_COST, FN_LOSS_MULTIPLIER = profit_params(config)

print(f"Net revenue if repaid:       ${INTEREST_REVENUE:,.0f}")
print(f"Net loss if defaulted:       ${LOSS_AMOUNT:,.0f}")
print(f"Servicing cost per loan:     ${SERVICE_COST:,.0f}")
print(f"Default loss multiplier:     {FN_LOSS_MULTIPLIER:.2f}x")


#### Load scored model + held-out test set

In [ ]:
model_path = Path("src/models/calibrated_model.joblib")
if not model_path.exists():
    model_path = Path("src/models/xgboost_model.joblib")

model = joblib.load(model_path)
X_test = pd.read_parquet("src/models/X_test.parquet")
y_test = pd.read_parquet("src/models/y_test.parquet")["default"]

# Use calibrated probabilities when available; otherwise fall back to the raw model.
p_default = model.predict_proba(X_test)[:, 1]
print(f"Model used: {model_path.name}")
print(f"Test set: {len(X_test):,} loans")
print(f"Avg predicted default probability: {p_default.mean():.1%}")


#### Calculate expected profit per loan:

In [ ]:
test_profits = expected_profit_per_loan(
    p_default,
    INTEREST_REVENUE,
    LOSS_AMOUNT,
    servicing_cost=SERVICE_COST,
    fn_loss_multiplier=FN_LOSS_MULTIPLIER,
)
realized_test_profits = realized_profit_per_loan(
    y_test.values,
    INTEREST_REVENUE,
    LOSS_AMOUNT,
    servicing_cost=SERVICE_COST,
    fn_loss_multiplier=FN_LOSS_MULTIPLIER,
)

results_df = X_test.copy()
results_df["p_default"]       = p_default
results_df["expected_profit"] = test_profits
results_df["realized_profit"] = realized_test_profits
results_df["actual_default"]  = y_test.values

print(f"Avg expected profit per loan: ${test_profits.mean():,.0f}")
print(f"Avg realized profit per loan: ${realized_test_profits.mean():,.0f}")
print(f"Loans with positive expected profit: {(test_profits > 0).mean():.1%}")
results_df[["p_default", "expected_profit", "realized_profit", "actual_default"]].head(10)

#### Profit curve

In [ ]:
thresholds = np.arange(0.01, 1.01, 0.01)
curve_df = portfolio_profit_curve(
    p_default,
    y_test.values,
    thresholds,
    INTEREST_REVENUE,
    LOSS_AMOUNT,
    servicing_cost=SERVICE_COST,
    fn_loss_multiplier=FN_LOSS_MULTIPLIER,
)

portfolio_profits = curve_df["portfolio_profit"].to_numpy()
approval_rates = curve_df["approval_rate"].to_numpy()
optimal_idx = np.argmax(portfolio_profits)
optimal_threshold = thresholds[optimal_idx]
max_profit = portfolio_profits[optimal_idx]
profit_at_05 = portfolio_profits[np.argmin(np.abs(thresholds - 0.5))]
profit_at_100 = curve_df.loc[np.isclose(curve_df["threshold"], 1.0), "portfolio_profit"].iloc[0]

print(f"Optimal threshold:         {optimal_threshold:.2f}")
print(f"Max portfolio profit:      ${max_profit:,.0f}")
print(f"Profit at 0.5 threshold:   ${profit_at_05:,.0f}")
print(f"Profit at 100% approval:   ${profit_at_100:,.0f}")
print(f"Profit improvement:        ${max_profit - profit_at_05:,.0f}")

#### Plot the profit curve

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(thresholds, portfolio_profits, color="#4C9BE8", linewidth=2.5, label="Portfolio profit")
ax1.axvline(optimal_threshold, color="#E8593C", linestyle="--", linewidth=1.5,
            label=f"Optimal threshold ({optimal_threshold:.2f})")
ax1.axvline(0.5, color="#888", linestyle=":", linewidth=1.5,
            label="Default threshold (0.50)")
ax1.axhline(0, color="black", linewidth=0.5, alpha=0.3)
ax1.set_xlabel("Decision threshold (approve if P(default) < threshold)")
ax1.set_ylabel("Total portfolio profit ($)")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))

ax2 = ax1.twinx()
ax2.plot(thresholds, approval_rates, color="#4CAF50", linewidth=1.5, 
         linestyle="-.", alpha=0.7, label="Approval rate")
ax2.set_ylabel("Approval rate", color="#4CAF50")
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower right")

ax1.set_title("Portfolio profit vs decision threshold")
plt.tight_layout()
plt.show()

### Save Profit Threshold

In [ ]:
os.makedirs("src/models", exist_ok=True)

# Model/test artifacts are produced earlier in the pipeline; save only the optimized threshold here.
joblib.dump(float(optimal_threshold), "src/models/optimal_threshold.joblib")

print(f"Threshold saved: {optimal_threshold:.2f}")
print(f"Test set referenced: {len(X_test):,} rows")
